![manufacturing gears](manufacturing.jpg)

Manufacturing processes for any product is like putting together a puzzle. Products are pieced together step by step, and keeping a close eye on the process is important.

For this project, you're supporting a team that wants to improve how they monitor and control a manufacturing process. The goal is to implement a more methodical approach known as statistical process control (SPC). SPC is an established strategy that uses data to determine whether the process works well. Processes are only adjusted if measurements fall outside of an acceptable range. 

This acceptable range is defined by an upper control limit (UCL) and a lower control limit (LCL), the formulas for which are:

$ucl = avg\_height + 3 * \frac{stddev\_height}{\sqrt{5}}$

$lcl = avg\_height - 3 * \frac{stddev\_height}{\sqrt{5}}$

The UCL defines the highest acceptable height for the parts, while the LCL defines the lowest acceptable height for the parts. Ideally, parts should fall between the two limits.

Using SQL window functions and nested queries, you'll analyze historical manufacturing data to define this acceptable range and identify any points in the process that fall outside of the range and therefore require adjustments. This will ensure a smooth running manufacturing process consistently making high-quality products.

## The data
The data is available in the `manufacturing_parts` table which has the following fields:
- `item_no`: the item number
- `length`: the length of the item made
- `width`: the width of the item made
- `height`: the height of the item made
- `operator`: the operating machine
    - There are 20 unique operators. They are formatted as: "Op-#".

There are 500 rows in the table.

# Objectives
- Analyze the manufacturing_parts table and determine whether the manufacturing process is performing within acceptable control limits:.
    - Create an alert that flags whether the height of a product is within the control limits for each `operator` using the formulas provided in the workbook.
        - The final query should return the following fields: `operator`, `row_number`, `height`, `avg_height`, `stddev_height`, `ucl`, `lcl`, `alert`, and be ordered by the `item_no`.
        - The `alert` column will be your boolean flag.
        - Use a window function of length 5 to calculate the control limits, considering rows up to and including the current row; incomplete windows should be excluded from the final query output. Save this DataFrame as `alerts`.

The code below follows PostGreSQL syntax. The cell directly below includes notes & drafts of code that will be used later for the final query.

In [ ]:
#In utilizing subqueries, can refer to calculations via their aliases rather than copying-pasting them multiple times

#First obtain new fields per operator: row_number, avg_height, stddev_height, lcl, ucl
    #order by 'item_no'
    #include 5 rows in each partition
SELECT operator,
    ROW_NUMBER() OVER (PARTITION BY operator ORDER BY item_no ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS row_number,
    height,
    AVG(height) OVER (PARTITION BY operator ORDER BY item_no ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS
        avg_height,
    STDDEV(height) OVER (PARTITION BY operator ORDER BY item_no ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS
        stddev_height

#Calculate ucl, lcl using these calculations via a subquery
    #since the sliding window relies on the previous 4 rows for each calculation, there is not enough data for the first 4 
        # data points to create some necessary features; exclude these utilizing the 'row_number' feature
SELECT subq1.*,
    (avg_height + 3 * ( (stddev_height) / (SQRT(5) ))) AS ucl,
    (avg_height - 3 * ( (stddev_height) / (SQRT(5) ))) AS lcl
FROM (
    SELECT operator,
        ROW_NUMBER() OVER (PARTITION BY operator ORDER BY item_no ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS row_number,
        height,
        AVG(height) OVER (PARTITION BY operator ORDER BY item_no ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS
            avg_height,
        STDDEV(height) OVER (PARTITION BY operator ORDER BY item_no ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS
            stddev_height
        FROM manufacturing_parts) AS subq1
WHERE subq1.row_number >= 5

#Need to instantiate an alert for when height < lcl OR height > ucl. Utilize & alias previous 2 queries via subqueries
SELECT subq2.*,
    CASE WHEN height < lcl OR height > ucl THEN TRUE
        ELSE FALSE END AS alert
FROM (
    SELECT subq1.*,
        (avg_height + 3 * ( (stddev_height) / (SQRT(5) ))) AS ucl,
        (avg_height - 3 * ( (stddev_height) / (SQRT(5) ))) AS lcl
    FROM (
        SELECT operator,
            ROW_NUMBER() OVER (PARTITION BY operator ORDER BY item_no ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS row_number,
            height,
            AVG(height) OVER (PARTITION BY operator ORDER BY item_no ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS
        avg_height,
            STDDEV(height) OVER (PARTITION BY operator ORDER BY item_no ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS
        stddev_height
        FROM manufacturing_parts) AS subq1
        WHERE subq1.row_number >= 5) AS subq2;

The cell below is the final SQL query.

In [2]:
#FINAL QUERY
    #420 rows
SELECT subq2.*,
    CASE WHEN height < lcl OR height > ucl THEN TRUE
        ELSE FALSE END AS alert
FROM (
    SELECT subq1.*,
        (avg_height + 3 * ( (stddev_height) / (SQRT(5) ))) AS ucl,
        (avg_height - 3 * ( (stddev_height) / (SQRT(5) ))) AS lcl
    FROM (
        SELECT operator,
            ROW_NUMBER() OVER (PARTITION BY operator ORDER BY item_no ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS row_number,
            height,
            AVG(height) OVER (PARTITION BY operator ORDER BY item_no ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS avg_height,
            STDDEV(height) OVER (PARTITION BY operator ORDER BY item_no ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS 
                    stddev_height
        FROM manufacturing_parts) AS subq1
    WHERE subq1.row_number >= 5) AS subq2;

This image shows the results of the above query.
![query_results.png](query_results.png)